Проверка сохранённых оценок трёх черновиков USC. Оценки восстанавливаются только при полном совпадении случая, крит.


In [1]:
from pathlib import Path
import hashlib
import json
import os
import re
import numpy as np
import pandas as pd
start = Path.cwd().resolve()
ROOT = next((p for p in (start, *start.parents) if (p / 'experiments/clinical_protocols').is_dir() and (p / 'data').is_dir()))
if os.name == 'nt' and (not str(ROOT).startswith(chr(92) * 2 + '?' + chr(92))):
    ROOT = Path(chr(92) * 2 + '?' + chr(92) + str(ROOT))
HERE = ROOT / 'experiments/token_budget_controls/universal_self_consistency_gpt_drafts_three_drafts'
USC = ROOT / 'experiments/token_budget_controls/universal_self_consistency_gpt_seven_samples'

def normalize(value):
    return '' if pd.isna(value) else str(value).replace('\r\n', '\n').replace('\r', '\n').strip()

def protocol_text(value):
    return re.split('(?im)^\\s*Protocol Author\\s*:', normalize(value), maxsplit=1)[0].strip()

def input_digest(vignette, criterion, text):
    serialized = json.dumps([normalize(vignette), normalize(criterion), protocol_text(text)], ensure_ascii=False)
    return hashlib.sha256(serialized.encode('utf-8')).hexdigest()


In [2]:
targets = []
for number in (1, 2, 3):
    raw = pd.read_csv(HERE / f'usc_gpt_draft{number}.csv', sep=';', encoding='utf-8-sig', low_memory=False)
    raw = raw[raw.Vignette.notna() & raw.winner_protocol.notna()]
    for _, row in raw.iterrows():
        targets.append({'draft_index': number, 'Vignette': normalize(row.Vignette), 'input_sha256': input_digest(row.Vignette, row.Criterion, row.winner_protocol)})
targets = pd.DataFrame(targets)
assert len(targets) == 687
assert not targets.duplicated(['draft_index', 'Vignette']).any()


In [3]:
saved = {}
for size in (3, 5, 7):
    path = USC / 'llm_as_judge' / f'usc_gpt_sc{size}_judge.csv'
    judged = pd.read_csv(path, sep=';', encoding='utf-8-sig', low_memory=False)
    judged = judged.dropna(subset=['Vignette', 'winner_protocol', 'Judge_score'])
    for _, row in judged.iterrows():
        digest = input_digest(row.Vignette, row.Criterion, row.winner_protocol)
        match = {'score': float(row.Judge_score), 'source': path.relative_to(ROOT).as_posix()}
        if match not in saved.setdefault(digest, []):
            saved[digest].append(match)
rows = []
for target in targets.to_dict('records'):
    matches = saved.get(target['input_sha256'], [])
    values = {match['score'] for match in matches}
    status = 'recovered_exact_match' if len(values) == 1 else 'requires_api_judgment' if not values else 'conflicting_saved_judgments'
    rows.append({**target, 'status': status, 'Judge_score': next(iter(values)) if len(values) == 1 else np.nan, 'source_judge_csv': ' | '.join(sorted({match['source'] for match in matches}))})
coverage = pd.DataFrame(rows)


In [4]:
summary = coverage.groupby('draft_index').agg(required=('status', 'size'), recovered=('status', lambda values: values.eq('recovered_exact_match').sum()), missing=('status', lambda values: values.eq('requires_api_judgment').sum()), conflicting=('status', lambda values: values.eq('conflicting_saved_judgments').sum()))
summary['coverage_fraction'] = summary.recovered / summary.required
display(summary)
complete_cases = coverage.groupby('Vignette').status.apply(lambda values: values.eq('recovered_exact_match').all())
print('Полностью оценённых троек:', int(complete_cases.sum()), 'из', len(complete_cases))
print('Недостающих оценок:', int(summary.missing.sum()))
if summary.missing.sum() or summary.conflicting.sum():
    print('Полное сравнение mean@3/max@3 ожидает недостающих судейских оценок.')


,required,recovered,missing,conflicting,coverage_fraction
draft_index,,,,,
1,229,130,99,0,0.567686
2,229,103,126,0,0.449782
3,229,103,126,0,0.449782


Полностью оценённых троек: 3 из 229
Недостающих оценок: 351
Полное сравнение mean@3/max@3 ожидает недостающих судейских оценок.


In [5]:
coverage.to_csv(HERE / 'draft_judgment_coverage.csv', sep=';', index=False, encoding='utf-8-sig')
summary.to_csv(HERE / 'draft_judgment_summary.csv', sep=';', encoding='utf-8-sig')
print('Сохранены draft_judgment_coverage.csv и draft_judgment_summary.csv')


Сохранены draft_judgment_coverage.csv и draft_judgment_summary.csv
